Step 1: Ingest and Check the File Path Layout

In [ ]:
import os

# Define the relative path to your raw data from the notebooks folder
data_folder = "../data/raw"

if os.path.exists(data_folder):
    csv_files = sorted([f for f in os.listdir(data_folder) if f.endswith('.csv')])
    print(f"✅ Connection successful! Found folder: {data_folder}")
    print(f"📁 Target files detected ({len(csv_files)} total):")
    for file in csv_files:
        print(f" - {file}")
else:
    print(f"❌ Error: Could not find the folder path at '{data_folder}'")
    print(f"Current working directory layout is: {os.getcwd()}")


Step 2: Read Data Profiles (.shape and .dtypes)

In [ ]:
import os
import pandas as pd

data_folder = "../data/raw"
csv_files = sorted([f for f in os.listdir(data_folder) if f.endswith('.csv')])

print("Starting deep dataset profile scan...\n")

for idx, file in enumerate(csv_files, start=1):
    path = os.path.join(data_folder, file)
    print("=" * 70)
    print(f"📊 DATASET {idx}: {file}")
    print("=" * 70)
    
    try:
        # Load file into an in-memory DataFrame
        df = pd.read_csv(path)
        
        # Task Requirement: Print Dimensions (.shape)
        print(f"📐 Structural Dimensions (Rows, Columns): {df.shape}")
        print("-" * 50)
        
        # Task Requirement: Print Data Formats (.dtypes)
        print("📋 Variable Column Mappings (.dtypes):")
        print(df.dtypes)
        
    except Exception as e:
        print(f"❌ Structural Read Error on file {file}: {e}")
        
    print("\n" + "_"*70 + "\n")


In [ ]:
#launch_date is showing str which is an anomaly since it should be a date. We will need to convert it to datetime format for proper analysis.


#Step 3: Print Previews (.head()) and Find Anomalies

In [ ]:
import os
import pandas as pd

data_folder = "../data/raw"
csv_files = sorted([f for f in os.listdir(data_folder) if f.endswith('.csv')])

print("Extracting row previews and scanning structural integrity...\n")

for idx, file in enumerate(csv_files, start=1):
    path = os.path.join(data_folder, file)
    print("=" * 70)
    print(f"🔍 INSPECTING DATASET {idx}: {file}")
    print("=" * 70)
    
    try:
        df = pd.read_csv(path)
        
        # Task Requirement: Print Sample Row Layout (.head)
        print("👋 First 3 Rows Preview (.head(3)):")
        display(df.head(3))
        print("-" * 50)
        
        # Task Requirement: Check for Anomalies (Null Values & Duplicates)
        null_counts = df.isnull().sum()
        total_nulls = null_counts.sum()
        duplicate_rows = df.duplicated().sum()
        
        print("🛠️ Integrity Diagnostics:")
        if total_nulls > 0:
            print(f"⚠️ Missing Data: Found {total_nulls} blank fields across the dataset.")
            print("Missing counts by column:")
            print(null_counts[null_counts > 0])
        else:
            print("✅ Data Integrity: Zero missing (NaN) values detected.")
            
        if duplicate_rows > 0:
            print(f"⚠️ Redundancy: Found {duplicate_rows} exact identical duplicate rows.")
        else:
            print("✅ Uniqueness: No duplicate records found.")
            
    except Exception as e:
        print(f"❌ Error inspecting file {file}: {e}")
        
    print("\n" + "_"*70 + "\n")


The Integrity Diagnostics , The diagnostic scanner evaluated the entire dataset to find missing values and duplicates, but it only printed out the first 3 rows (.head(3)) for your visual preview. This is a common practice to avoid overwhelming the output with too much data while still giving you a snapshot of the dataset's structure and content. The integrity diagnostics will still report the total counts of missing values and duplicates for the entire dataset, even though only a few rows are shown in the preview.
The integrity diagnostics counts (df.isnull().sum()) look at every single cell from the first row to the absolute last row of each file to ensure no hidden blanks are missed

Step 4: Fetch Live NAV from the Web API

In [ ]:
import os
import requests
import pandas as pd

# Target directory to store your fetched files
output_dir = "../data/raw"
os.makedirs(output_dir, exist_ok=True)

schemes = {
    "125497": "HDFC_Top_100_Direct",
    "119551": "SBI_Bluechip",
    "120503": "ICICI_Bluechip",
    "118632": "Nippon_Large_Cap",
    "119092": "Axis_Bluechip",
    "120841": "Kotak_Bluechip"
}

print("Running API Fetcher with explicit path structure...\n")

for code, name in schemes.items():
    # Constructing URL with explicit slash character separation
    url = "https://mfapi.in" + str(code)
    print(f"📡 Calling: {url}")
    
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            json_data = response.json()
            nav_records = json_data.get('data', [])
            
            df_live = pd.DataFrame(nav_records)
            if not df_live.empty:
                df_live['scheme_code'] = code
                df_live['scheme_name'] = name
                df_live = df_live[['scheme_code', 'scheme_name', 'date', 'nav']]
                
                file_name = f"live_nav_{code}_{name.lower()}.csv"
                df_live.to_csv(os.path.join(output_dir, file_name), index=False)
                print(f"✅ Success: Downloaded {len(df_live)} rows to {file_name}")
            else:
                print(f"⚠️ Empty dataset returned for {code}")
        else:
            print(f"❌ HTTP Server Error: Status {response.status_code}")
    except Exception as e:
        print(f"❌ Connection Crash: {e}")
    print("-" * 60)


Step 4: Paste the Fixed API Code Block

In [1]:
import os
import requests
import pandas as pd

# Hardcoding the destination folder path explicitly
output_dir = "../data/raw"
os.makedirs(output_dir, exist_ok=True)

# A clean mapping tracking our targets explicitly
schemes = {
    "125497": "HDFC_Top_100_Direct",
    "119551": "SBI_Bluechip",
    "120503": "ICICI_Bluechip",
    "118632": "Nippon_Large_Cap",
    "119092": "Axis_Bluechip",
    "120841": "Kotak_Bluechip"
}

print("Initiating API connection with direct path string mapping...\n")

for code, name in schemes.items():
    # Direct hardcoded path generation to completely bypass variables memory cache
    url = f"https://api.mfapi.in/mf/{code}"
    print(f"📡 Requesting: {url}")
    
    try:
        response = requests.get(url, timeout=12)
        if response.status_code == 200:
            json_data = response.json()
            nav_records = json_data.get('data', [])
            
            df_live = pd.DataFrame(nav_records)
            if not df_live.empty:
                df_live['scheme_code'] = code
                df_live['scheme_name'] = name
                df_live = df_live[['scheme_code', 'scheme_name', 'date', 'nav']]
                
                file_name = f"live_nav_{code}_{name.lower()}.csv"
                df_live.to_csv(os.path.join(output_dir, file_name), index=False)
                print(f"✅ Success: Downloaded {len(df_live)} rows.")
            else:
                print(f"⚠️ Empty response payload array returned for code: {code}")
        else:
            print(f"❌ Server Connection Error: HTTP Code {response.status_code}")
    except Exception as error:
        print(f"❌ Connection Failure: {error}")
    print("-" * 55)

print("\nAPI tasks sequence finished.")


Initiating API connection with direct path string mapping...

📡 Requesting: https://api.mfapi.in/mf/125497
✅ Success: Downloaded 3092 rows.
-------------------------------------------------------
📡 Requesting: https://api.mfapi.in/mf/119551
✅ Success: Downloaded 3237 rows.
-------------------------------------------------------
📡 Requesting: https://api.mfapi.in/mf/120503
✅ Success: Downloaded 3308 rows.
-------------------------------------------------------
📡 Requesting: https://api.mfapi.in/mf/118632
✅ Success: Downloaded 3299 rows.
-------------------------------------------------------
📡 Requesting: https://api.mfapi.in/mf/119092
✅ Success: Downloaded 3566 rows.
-------------------------------------------------------
📡 Requesting: https://api.mfapi.in/mf/120841
✅ Success: Downloaded 3302 rows.
-------------------------------------------------------

API tasks sequence finished.


I was going wrong here as my previous code had wrong url
Hence i corrected it using ai mode in google & went through the code to identify where i went wrong
even after correcting the code vs cpde was running the prev code saved in its memory cache
So i restarted the py kernelfrom the top menu bar , then clicked clear nall outputs from the menu bar
This cleared all prev o/ps & restarted the kernel later the corrected code executed successfully

Explore fund master — print unique fund houses, categories, sub-categories, risk grades. Understand AMFI scheme code structure.
Validate AMFI codes — confirm every code in fund_master exists in nav_history. Write a short data quality summary.

In [3]:
import os
import pandas as pd

master_file_path = "../data/raw/01_fund_master.csv"

if os.path.exists(master_file_path):
    print("🎯 Loading Fund Master Records Matrix...\n")
    df_master = pd.read_csv(master_file_path)
    
    # Define columns explicitly requested in your internship brief
    target_columns = ['fund_house', 'category', 'sub_category', 'risk_category']
    
    for col in target_columns:
        # Check for alternative spellings (like risk_grade vs risk_category)
        actual_col = col if col in df_master.columns else [c for c in df_master.columns if col[:4] in c.lower()]
        actual_col = actual_col[0] if isinstance(actual_col, list) and actual_col else col
        
        if actual_col in df_master.columns:
            unique_items = df_master[actual_col].dropna().unique()
            print("=" * 65)
            print(f"📊 COLUMN: '{actual_col}' | Total Unique Elements: {len(unique_items)}")
            print("=" * 65)
            print(sorted([str(x) for x in unique_items]))
            print("\n")
        else:
            print(f"⚠️ Column hint '{col}' could not be matched explicitly against data fields: {list(df_master.columns)}\n")
            
else:
    print(f"❌ Error: Could not locate the file target layout at {master_file_path}")


🎯 Loading Fund Master Records Matrix...

📊 COLUMN: 'fund_house' | Total Unique Elements: 10
['Aditya Birla Sun Life MF', 'Axis Mutual Fund', 'DSP Mutual Fund', 'HDFC Mutual Fund', 'ICICI Prudential MF', 'Kotak Mahindra MF', 'Mirae Asset MF', 'Nippon India MF', 'SBI Mutual Fund', 'UTI Mutual Fund']


📊 COLUMN: 'category' | Total Unique Elements: 2
['Debt', 'Equity']


📊 COLUMN: 'sub_category' | Total Unique Elements: 12
['ELSS', 'Flexi Cap', 'Gilt', 'Index', 'Index/ETF', 'Large & Mid Cap', 'Large Cap', 'Liquid', 'Mid Cap', 'Short Duration', 'Small Cap', 'Value']


📊 COLUMN: 'risk_category' | Total Unique Elements: 5
['High', 'Low', 'Moderate', 'Moderately High', 'Very High']




An AMFI scheme code is a unique 5- or 6-digit identification number assigned to individual mutual fund products in India. 

Step 8: Execute AMFI Code Structural Cross-Validation Check

In [4]:
import os
import pandas as pd

master_path = "../data/raw/01_fund_master.csv"
history_path = "../data/raw/02_nav_history.csv"

if os.path.exists(master_path) and os.path.exists(history_path):
    print("⏳ Running comprehensive cross-dataset relational validation layers...\n")
    
    df_master = pd.read_csv(master_path)
    df_history = pd.read_csv(history_path)
    
    # Standardize and pull out raw code sets
    # (Matches common variants like amfi_code or scheme_code)
    m_col = 'amfi_code' if 'amfi_code' in df_master.columns else ('scheme_code' if 'scheme_code' in df_master.columns else df_master.columns[0])
    h_col = 'amfi_code' if 'amfi_code' in df_history.columns else ('scheme_code' if 'scheme_code' in df_history.columns else df_history.columns[0])
    
    print(f"🔗 Mapping column '{m_col}' in Fund Master to column '{h_col}' in NAV History...")
    
    master_codes = set(df_master[m_col].dropna().unique())
    history_codes = set(df_history[h_col].dropna().unique())
    
    # Set difference math: items in Master that are missing from History
    missing_in_history = master_codes - history_codes
    
    print("=" * 65)
    print("📋 CODES RELATION MATRIX STATUS")
    print("=" * 65)
    print(f"• Unique codes in Fund Master dataset list:  {len(master_codes)}")
    print(f"• Unique codes tracked inside NAV History:    {len(history_codes)}")
    print("-" * 65)
    
    if len(missing_in_history) == 0:
        print("✅ CROSS-VALIDATION PASSED PERFECTLY!")
        print("Every single AMFI scheme code listed in your master catalog file has perfectly matching historical NAV logs.")
    else:
        print(f"⚠️ DATA QUALITY ANOMALY DETECTED!")
        print(f"Found {len(missing_in_history)} AMFI scheme codes in Fund Master that have ZERO records in NAV History.")
        print(f"❌ Missing scheme code targets list: {sorted(list(missing_in_history))}")
        
else:
    print("❌ Script mapping aborted: Check that both '01_fund_master.csv' and '02_nav_history.csv' sit inside '../data/raw/'")


⏳ Running comprehensive cross-dataset relational validation layers...

🔗 Mapping column 'amfi_code' in Fund Master to column 'amfi_code' in NAV History...
📋 CODES RELATION MATRIX STATUS
• Unique codes in Fund Master dataset list:  40
• Unique codes tracked inside NAV History:    40
-----------------------------------------------------------------
✅ CROSS-VALIDATION PASSED PERFECTLY!
Every single AMFI scheme code listed in your master catalog file has perfectly matching historical NAV logs.


Net Asset Value (NAV)NAV is the price of a single unit of a mutual fund. Think of it as the stock price for mutual funds. It tells you how much one share of the fund is worth today.The Formula:\(\text{NAV}=\frac{\text{Total\ Assets\ of\ the\ Fund}-\text{Total\ Liabilities}}{\text{Total\ Number\ of\ Outstanding\ Units}}\)Simple ExampleImagine a mutual fund owns ₹1,00,000 worth of stocks. The fund owes ₹5,000 in management fees. It has issued a total of 1,000 units to investors.Calculation: ₹1,00,000 minus ₹5,000 leaves ₹95,000. Divide ₹95,000 by 1,000 units.The Result: The NAV is ₹95. If you want to buy 1 unit of this fund today, it will cost you ₹95.

An AMFI Code is a unique 6-digit identification number given to every mutual fund scheme in India. Think of it exactly like a roll number in a school or a barcode on a product.
Instead of typing long names like "Axis Bluechip Fund - Direct Plan - Growth Option", financial systems use its unique AMFI code (120503) to identify it instantly without any typos.

Data Quality Summary Report: Day 1 Ingestion1.
 Structural Footprint & CompletenessFile Ingestion Status: 
 All 10 baseline raw CSV datasets were loaded into memory via Pandas with zero structural reading crashes.
 Dimensions & Completeness: The core dimensional maps are intact 
 (e.g., 01_fund_master.csv contains 40 schemes across 15 descriptive attributes).Missing (Null) Values:
The baseline structural identifier arrays contain zero critical null entries.
Observation: Optional operational fields in informational tables present minor missing value parameters which
 do not disrupt database keys.2.
  Primary Key & Uniqueness VerificationData Redundancy Check:
Running deduplication evaluations (df.duplicated().sum()) confirmed zero duplicate rows across the baseline datasets.
Key Uniqueness: Core primary indexing constraints are structurally sound and exhibit pure statistical unique counts.
3. Data Type & Schema Alignment AnomaliesDate Parsing Latency: 
The variable attribute column mappings (.dtypes) flagged that launch_date and 
timestamp inputs are stored as plain text strings (object type).
Impact: These columns must be cast using pd.to_datetime() during the Day 2 transformation pipeline before
 running chronological filters or rolling window calculations.Numeric Strings: Numerical transaction codes and
  quantitative parameters are cleanly aligned as integers (int64) or fractional decimals (float64).
  4. Cross-Dataset Relational IntegrityAMFI Code Referencing: The set-difference cross-validation diagnostic confirmed
   that every active AMFI scheme code listed in the 01_fund_master.csv master catalog file maps cleanly to 
   corresponding timestamp data logs within 02_nav_history.csv.Relational Passing Status: 100% Passed.
   There are no orphan keys or disconnected relational rows across your structural schemas.

For git push used these cmds step by step on vs cpde terminal:
Step 0: Check Your Staging Area
git status (op in red)

Step 1: Stage All Your FilesWe need to change those file names from red (untracked) to green (staged).
git add .
Once you press enter, type git status one more time to make sure those file names turned green. Reply with next once you see the green text, or let me know if any message pops up!

Step 2: Verify Your Staged Files
git status
Check the text that prints out. You should now see all your files listed in green text under a heading that says "Changes to be committed".

Step 3: Run the Commit Command
git commit -m "Day 1: Data ingestion complete"

STEP : Push ur work on github profile
Your local repository is now successfully linked to your online cloud repository. Let's execute the absolute final command to upload your Day 1 internship progress.Step 3: Push Your Code onlineType this exact command into your terminal and press Enter:
powershell
git push -u origin main
